In [1]:
import os
from google.colab import drive

print("[INFO] Connecting to Google Drive...")
drive.mount('/content/drive')
os.chdir('/content')
# Remove any existing repo and clone the official GRU4Rec implementation
!rm -rf /content/GRU4Rec_PyTorch_Official
!git clone https://github.com/hidasib/GRU4Rec_PyTorch_Official.git
# patching the repository
file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    code = file.read()

code = code.replace("torch.tensor(np.vstack(m)", "torch.tensor(np.vstack(m), dtype=torch.float32")
code = code.replace("torch.tensor(np.vstack(m2)", "torch.tensor(np.vstack(m2), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b)", "torch.tensor(np.hstack(b), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b2)", "torch.tensor(np.hstack(b2), dtype=torch.float32")
code = code.replace(
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), device=self.Wy.weight.device)",
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), dtype=torch.float32, device=self.Wy.weight.device)"
)

with open(file_path, 'w') as file:
    file.write(code)

print("[SUCCESS] Base architecture patched and isolated!")


[INFO] Connecting to Google Drive...
Mounted at /content/drive
Cloning into 'GRU4Rec_PyTorch_Official'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 75 (delta 20), reused 15 (delta 15), pack-reused 48 (from 1)
Receiving objects: 100% (75/75), 362.75 KiB | 1.80 MiB/s, done.
Resolving deltas: 100% (35/35), done.
[SUCCESS] Base architecture patched and isolated!


In [2]:
import sys
import pickle
import torch

# load model
sys.path.append('/content/GRU4Rec_PyTorch_Official')
import gru4rec_pytorch

print("[INFO] Loading the Black-Box Oracle (Victim)...")
load_path = '/content/drive/MyDrive/ML_Security_Project/victim_gru4rec.pkl'

with open(load_path, 'rb') as f:
    oracle = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
oracle.model.eval()
print(f"[SUCCESS] Black-Box Oracle is active and running on: {device}")


[INFO] Loading the Black-Box Oracle (Victim)...
[SUCCESS] Black-Box Oracle is active and running on: cuda


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# extract the basic structural constraints (depth and hidden dimensions)
NUM_LAYERS = len(oracle.layers)
HIDDEN_DIM = oracle.layers[-1]
print(f"[INFO] Oracle depth: {NUM_LAYERS} GRU layer(s), hidden size: {HIDDEN_DIM}")

# vocabulary alignment
_oracle_itemidmap = oracle.data_iterator.itemidmap
ATTACKER_CATALOG = sorted(_oracle_itemidmap.index.values.tolist())
item2idx = {raw_id: i for i, raw_id in enumerate(ATTACKER_CATALOG)}
idx2item = {i: raw_id for raw_id, i in item2idx.items()}
VOCAB_SIZE = len(ATTACKER_CATALOG)
print(f"[INFO] Catalog size (VOCAB_SIZE): {VOCAB_SIZE}")

_align = np.array([_oracle_itemidmap.loc[idx2item[i]] for i in range(VOCAB_SIZE)], dtype=np.int64)
ALIGN_ATTACKER_TO_ORACLE = torch.tensor(_align, dtype=torch.long, device=device)

@torch.no_grad()
def query_blackbox(attacker_idx_seq):
    """
    attacker_idx_seq: LongTensor (batch, seq_len), our own canonical indices.
    Returns softmax probs (batch, VOCAB_SIZE) in the SAME attacker order.
    Only function that touches oracle.model.
    """
    oracle_idx_seq = ALIGN_ATTACKER_TO_ORACLE[attacker_idx_seq]
    b, L = oracle_idx_seq.shape
    H = [torch.zeros(b, oracle.model.layers[i], device=device) for i in range(NUM_LAYERS)]
    logits = None
    for t in range(L):
        logits = oracle.model.forward(oracle_idx_seq[:, t], H, None, training=False)
    logits_attacker_order = torch.empty_like(logits)
    logits_attacker_order[:, ALIGN_ATTACKER_TO_ORACLE] = logits
    probs = F.softmax(logits_attacker_order, dim=1)
    return probs, logits_attacker_order

class SurrogateModel(nn.Module):
  """
    The Attacker's extraction model. We only match the victim's depth and vocabulary size.
    We initialize our own weights from scratch[cite: 5].
  """
    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hid_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, lengths=None):
        embedded = self.embedding(x)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.gru(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
            last_idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, out.size(2))
            last_hidden = out.gather(1, last_idx).squeeze(1)
        else:
            out, _ = self.gru(embedded)
            last_hidden = out[:, -1, :]
        return self.fc(last_hidden)

surrogate = SurrogateModel(VOCAB_SIZE, emb_dim=HIDDEN_DIM, hid_dim=HIDDEN_DIM, num_layers=NUM_LAYERS).to(device)
print("\n[SUCCESS] True black-box attack architecture initialized (no weight access).")


[INFO] Oracle depth: 1 GRU layer(s), hidden size: 100
[INFO] Catalog size (VOCAB_SIZE): 37526

[SUCCESS] True black-box attack architecture initialized (no weight access).


In [4]:
import pandas as pd

print("[INFO] Loading real (unlabeled) session prefixes to use as query prompts...")
train_data_path = '/content/drive/MyDrive/ML_Security_Project/yoochoose_train.csv'
train_df = pd.read_csv(train_data_path, sep='\t')

train_df['ItemIdx'] = train_df['ItemId'].map(item2idx)
train_df = train_df.dropna(subset=['ItemIdx'])
train_df['ItemIdx'] = train_df['ItemIdx'].astype(int)

_grouped = train_df.groupby('SessionId')['ItemIdx'].apply(list)
query_sessions = [s for s in _grouped if len(s) >= 2]
print(f"[SUCCESS] {len(query_sessions)} real session prefixes available as query prompts.")

def sample_query_batch(sessions, batch_size, length):
    """Sample `batch_size` random contiguous windows of exactly `length`
    items each, drawn from real sessions. Only used to build the INPUT to
    query_blackbox() -- never the ground-truth label."""
    eligible = [s for s in sessions if len(s) >= length]
    if not eligible:
        return None
    chosen = [eligible[i] for i in np.random.randint(0, len(eligible), size=batch_size)]
    out = np.empty((batch_size, length), dtype=np.int64)
    for i, s in enumerate(chosen):
        start = np.random.randint(0, len(s) - length + 1)
        out[i] = s[start:start + length]
    return torch.tensor(out, dtype=torch.long, device=device)


[INFO] Loading real (unlabeled) session prefixes to use as query prompts...
[SUCCESS] 7214493 real session prefixes available as query prompts.


In [5]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

epochs = 150
batch_size = 128
synthetic_batches_per_epoch = 50
max_seq_len = 10          # upper bound on query length; real length varies per batch now
temperature = 3.0

optimizer = optim.Adam(surrogate.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
criterion = nn.KLDivLoss(reduction='none')

print("[INFO] Starting black-box distillation using real session prefixes as query prompts...")
surrogate.train()
for epoch in range(epochs):
    total_loss = 0.0
    loop = tqdm(range(synthetic_batches_per_epoch), leave=False)

    for _ in loop:
        # Vary the query length batch-to-batch -> surrogate sees short AND
        # long sessions during training, matching real evaluation sessions
        L = np.random.randint(2, max_seq_len + 1)
        query_seqs = sample_query_batch(query_sessions, batch_size, L)
        if query_seqs is None:
            continue

        optimizer.zero_grad()

        oracle_soft_labels, oracle_raw_logits = query_blackbox(query_seqs)

        with torch.no_grad():
            max_probs, _ = torch.max(oracle_soft_labels, dim=1)
            weights = (max_probs / (max_probs.mean() + 1e-8)).unsqueeze(1)

        # get Surrogate predictions and apply temperature scaling
        surrogate_logits = surrogate(query_seqs)
        surrogate_log_probs = F.log_softmax(surrogate_logits / temperature, dim=1)
        oracle_soft_labels_t = F.softmax(oracle_raw_logits / temperature, dim=1)

        kl_raw = criterion(surrogate_log_probs, oracle_soft_labels_t)
        weighted_loss = (kl_raw * weights).sum(dim=1).mean() * (temperature ** 2)

        weighted_loss.backward()
        optimizer.step()
        total_loss += weighted_loss.item()

    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_epoch_loss = total_loss / synthetic_batches_per_epoch
        print(f"[INFO] Epoch {epoch+1:03d}/{epochs} | Weighted KL Loss: {avg_epoch_loss:.4f}")

print("\n[SUCCESS] Distillation finished.")


[INFO] Starting black-box distillation using real session prefixes as query prompts...


[INFO] Epoch 001/150 | Weighted KL Loss: 0.0348


[INFO] Epoch 010/150 | Weighted KL Loss: 0.0237


[INFO] Epoch 020/150 | Weighted KL Loss: 0.0211


[INFO] Epoch 030/150 | Weighted KL Loss: 0.0185


...

[INFO] Epoch 110/150 | Weighted KL Loss: 0.0130


[INFO] Epoch 120/150 | Weighted KL Loss: 0.0129


[INFO] Epoch 130/150 | Weighted KL Loss: 0.0132


[INFO] Epoch 140/150 | Weighted KL Loss: 0.0129


[INFO] Epoch 150/150 | Weighted KL Loss: 0.0132

[SUCCESS] Distillation finished.


In [6]:
print("[INFO] Initializing Evaluation Module (Model Fidelity & Agreement)...")

@torch.no_grad()
def calculate_agreement_at_k(surrogate_model, num_tests=500, k_list=(10, 20)):
    surrogate_model.eval()
    agreement = {k: 0.0 for k in k_list}
    max_k = max(k_list)

    for _ in range(num_tests):
        seq_len = np.random.randint(2, max_seq_len)
        seq = torch.randint(0, VOCAB_SIZE, (1, seq_len), device=device)

        surrogate_logits = surrogate_model(seq)
        _, surrogate_top_k = torch.topk(surrogate_logits.squeeze(0), max_k)
        surrogate_items = surrogate_top_k.cpu().numpy()

        oracle_probs, _ = query_blackbox(seq)
        _, oracle_top_k = torch.topk(oracle_probs.squeeze(0), max_k)
        oracle_items = oracle_top_k.cpu().numpy()

        for k in k_list:
            inter = len(set(surrogate_items[:k]) & set(oracle_items[:k]))
            agreement[k] += inter / k

    return {k: v / num_tests for k, v in agreement.items()}

print("[INFO] Running evaluation on 500 test sessions. Please wait...")
agr = calculate_agreement_at_k(surrogate, num_tests=500, k_list=(10, 20))

print("\n" + "="*50)
print("ATTACK METRICS (Fidelity / Agreement)")
print(f" -> Agreement@10 (Agr@10): {agr[10]:.4f}")
print(f" -> Agreement@20 (Agr@20): {agr[20]:.4f}")


[INFO] Initializing Evaluation Module (Model Fidelity & Agreement)...
[INFO] Running evaluation on 500 test sessions. Please wait...

ATTACK METRICS (Fidelity / Agreement)
 -> Agreement@10 (Agr@10): 0.0390
 -> Agreement@20 (Agr@20): 0.0497


In [7]:
from tqdm import tqdm

print("[INFO] Loading real dataset for Utility Evaluation (Multi-K)...")

try:
    df_test = pd.read_csv('/content/drive/MyDrive/ML_Security_Project/yoochoose_test.csv', sep='\t')
    df_test['ItemId'] = df_test['ItemId'].map(item2idx)
    df_test = df_test.dropna(subset=['ItemId'])
    df_test['ItemId'] = df_test['ItemId'].astype(int)
except FileNotFoundError as e:
    print(f"[ERROR] {e}. Please ensure the data paths are correct.")
    df_test = pd.DataFrame(columns=['SessionId', 'ItemId'])

def build_sessions(df, max_len=10):
    if df.empty:
        return []
    grouped = df.groupby('SessionId')['ItemId'].apply(list)
    sessions = []
    for session in grouped:
        if len(session) >= 2:
            target_item = session[-1]
            session_seq = session[:-1][-max_len:]
            sessions.append((session_seq, target_item))
    return sessions

eval_sessions = build_sessions(df_test)
print(f"[SUCCESS] Test environment ready with {len(eval_sessions)} valid sessions.")

@torch.no_grad()
def evaluate_utility_multi_k(surrogate_model, sessions, k_list=(1, 5, 10, 20), batch_size=1024):
    if len(sessions) == 0:
        print("[WARNING] No sessions to evaluate.")
        return

    surrogate_model.eval()
    total_recall = {k: 0.0 for k in k_list}
    total_ndcg = {k: 0.0 for k in k_list}
    max_k = max(k_list)
    n = len(sessions)

    for start in tqdm(range(0, n, batch_size), desc="Evaluating", unit="batch"):
        batch = sessions[start:start + batch_size]
        lengths = torch.tensor([len(s) for s, _ in batch], dtype=torch.long)
        maxlen = lengths.max().item()
        padded = torch.zeros(len(batch), maxlen, dtype=torch.long)
        for i, (seq, _) in enumerate(batch):
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        padded = padded.to(device)
        lengths = lengths.to(device)

        logits = surrogate_model(padded, lengths=lengths)
        _, top_k_tensor = torch.topk(logits, max_k, dim=1)
        top_k_items = top_k_tensor.cpu().numpy()

        for i, (_, target) in enumerate(batch):
            row = top_k_items[i]
            where = np.where(row == target)[0]
            if len(where):
                rank = where[0]
                for k in k_list:
                    if rank < k:
                        total_recall[k] += 1.0
                        total_ndcg[k] += 1.0 / np.log2(rank + 2)

    print("\n" + "="*60)
    print("REAL-WORLD UTILITY METRICS (Surrogate vs Ground Truth)")
    print("="*60)
    for k in sorted(k_list):
        print(f" -> Recall@{k:02d}: {total_recall[k]/n:.4f}  |  NDCG@{k:02d}: {total_ndcg[k]/n:.4f}")
    print("="*60)

if eval_sessions:
    evaluate_utility_multi_k(surrogate, eval_sessions, k_list=[1, 5, 10, 20])


[INFO] Loading real dataset for Utility Evaluation (Multi-K)...
[SUCCESS] Test environment ready with 733813 valid sessions.


Evaluating: 100%|██████████| 717/717 [00:19<00:00, 36.27batch/s]


REAL-WORLD UTILITY METRICS (Surrogate vs Ground Truth)
 -> Recall@01: 0.0001  |  NDCG@01: 0.0001
 -> Recall@05: 0.0003  |  NDCG@05: 0.0002
 -> Recall@10: 0.0007  |  NDCG@10: 0.0003
 -> Recall@20: 0.0021  |  NDCG@20: 0.0007
